In [1]:
from ultralytics import YOLO
import yaml
import pandas as pd
import torch
import os
import gc
import shutil
torch.cuda.set_per_process_memory_fraction(0.90, device=0) # limiting so I can use my pc
#clear everything just in case
gc.collect()
torch.cuda.empty_cache()

In [2]:
# ──── Augmentation & Hyperparameter & Name Configs ────
augmentation_config = 'YAMLs/Augmentation/medium_augmentation.yaml'
hyperparams_config = 'YAMLs/Hyperparameter/spec_AdamW.yaml'
base_model_path = 'BaseModels/yolo11m.pt' # change this to change the base model

model_name = os.path.basename(base_model_path).split('.')[0]
run_name = f"{model_name}-{augmentation_config.split('/')[-1].split('_')[0]}-{hyperparams_config.split('/')[-1].split('.')[0]}"

# need these for future to check the run progress
checkpoint_path = f"runs/detect/{run_name}/weights/last.pt"
weights_dir = os.path.dirname(checkpoint_path)

print(f"{'='*60}")
print(f"   Run Name: {run_name}")
print(f"{'='*60}\n")

   Run Name: yolo11m-medium-spec_AdamW



In [3]:
# ──── Load Configs ────
def load_yaml_config(filepath): # simple yaml loader
    with open(filepath, 'r') as f:
        return yaml.safe_load(f)

augmentation_config_dict = load_yaml_config(augmentation_config)
hyperparams_config_dict = load_yaml_config(hyperparams_config)
training_config = { # full arguments that I will push directly into YOLO since it can handle it
    **augmentation_config_dict,
    **hyperparams_config_dict
}

print(f"   Configuration loaded:")
print(f"   ├─ Augmentation: {augmentation_config.split('/')[-1]}")
print(f"   └─ Hyperparameters: {hyperparams_config.split('/')[-1]}\n")

   Configuration loaded:
   ├─ Augmentation: medium_augmentation.yaml
   └─ Hyperparameters: spec_AdamW.yaml



In [ ]:
# ──── Chunk & Epoch ────
# setting chunks, epoch and patience here because a single epoch takes around 10 to 13 minutes
# so its not possible for me to train it on one sitting
EPOCHS_PER_CHUNK = 15
TOTAL_EPOCHS = 500
PATIENCE = 50

# ultraytics doesnt have the patience tracking on non resumed training so I am tracking it myself with this yaml file
EARLY_STOP_TRACKER = f"runs/detect/{run_name}/early_stop_state.yaml"
# when a chunk is finished the last.pt will be stripped from the epochs variable (it will be set to -1) meaning I would have to start 
# the training from the last saved epoch so it is pretty much wasted, this yaml file track the true epochs
CHUNK_STATE_FILE = f"runs/detect/{run_name}/chunk_state.yaml"

print(f"{'='*60}")
print(f"   TRAINING IN CHUNKS")
print(f"   ├─ Epochs per chunk: {EPOCHS_PER_CHUNK}")
print(f"   ├─ Total target:     {TOTAL_EPOCHS}")
print(f"   └─ Patience:         {PATIENCE}")
print(f"{'='*60}\n")



   TRAINING IN CHUNKS
   ├─ Epochs per chunk: 15
   ├─ Total target:     500
   └─ Patience:         50



In [ ]:
# ──── Helper Functions ────
def get_checkpoint_epoch(path): 
    # get the epoch number from a checkpoint, returns -1 if not found or invalid
    checkpoint = torch.load(path, weights_only=False, map_location='cpu')
    return checkpoint.get('epoch', -1)

def save_early_stop_state(epochs_no_improve, best_fitness): 
    # save the patience tracking state to a yaml file so it can be loaded on resume
    os.makedirs(os.path.dirname(EARLY_STOP_TRACKER), exist_ok=True)
    with open(EARLY_STOP_TRACKER, 'w') as f:
        yaml.dump({'epochs_no_improve': epochs_no_improve, 'best_fitness': best_fitness}, f)

def load_early_stop_state(): 
    # load the patience tracking state from the yaml file, returns (0, -1.0) if not found
    if os.path.exists(EARLY_STOP_TRACKER):
        data = yaml.safe_load(open(EARLY_STOP_TRACKER))
        return data.get('epochs_no_improve', 0), data.get('best_fitness', -1.0)
    return 0, -1.0

def save_chunk_state(last_completed_epoch):
    # save the true epochs completed so that I can resume training from the last epoch
    os.makedirs(os.path.dirname(CHUNK_STATE_FILE), exist_ok=True)
    with open(CHUNK_STATE_FILE, 'w') as f:
        yaml.dump({'last_completed_epoch': last_completed_epoch}, f)

def load_chunk_state():
    # load the last completed epoch from the yaml file, returns -1 if not found
    if os.path.exists(CHUNK_STATE_FILE):
        with open(CHUNK_STATE_FILE) as f:
            return yaml.safe_load(f).get('last_completed_epoch', -1)
    return -1

def fix_chunk_artifacts(chunk_start_epoch, chunk_end_epoch):
    # after each chunk I need to rename the checkpoints and merge the results.csv to maintain a continuous log of epochs and results
    # tho I did add this a bit too late so I have some chunks with the artifacts but I will just ignore those since they are not that important
    # I will make sure to use this function for all future chunks
    chunk_size  = chunk_end_epoch - chunk_start_epoch
    results_csv = f"runs/detect/{run_name}/results.csv"
    master_csv  = f"runs/detect/{run_name}/results_master.csv"

    # ──── CSV merge ────
    if os.path.exists(results_csv): # if results.csv exists, merge it into master_csv with corrected epoch numbers
        new_df = pd.read_csv(results_csv)
        new_df.columns = new_df.columns.str.strip()       # YOLO pads col names
        if 'epoch' in new_df.columns:
            new_df['epoch'] = new_df['epoch'] + chunk_start_epoch

        if os.path.exists(master_csv): # if master_csv exists, append new_df to it, otherwise new_df becomes the master
            master_df = pd.read_csv(master_csv)
            master_df.columns = master_df.columns.str.strip()
            combined = pd.concat([master_df, new_df], ignore_index=True)
        else: # no master yet, new_df is the master
            combined = new_df

        combined.to_csv(master_csv, index=False) # this is the full log of all epochs, I will keep it even if results.csv gets deleted or corrupted
        combined.to_csv(results_csv, index=False)         # keep results.csv full too
        print(f"   CSV: +{len(new_df)} rows → {len(combined)} total epochs logged")

    # ──── Checkpoint rename ────
    # after each chunk the checkpoints are saved with local epoch names (epoch1.pt, epoch2.pt, etc.) but I want them to reflect the global epoch numbers (epoch16.pt, epoch17.pt, etc.) so I need to rename them
    # I also want to avoid overwriting checkpoints if they already exist, so I will skip renaming if the target name already exists, but I will always remove the local-named file to clean up the directory
    renamed, skipped = [], []
    if os.path.exists(weights_dir): # if weights_dir exists, rename the checkpoints from local epoch names to global epoch names
        for fname in sorted(os.listdir(weights_dir)):
            if fname.startswith('epoch') and fname.endswith('.pt'):
                try:
                    local_ep = int(fname.replace('epoch', '').replace('.pt', ''))
                except ValueError:
                    continue
                if local_ep <= chunk_size:
                    src      = os.path.join(weights_dir, fname)
                    global_ep = chunk_start_epoch + local_ep
                    new_name  = f'epoch{global_ep}.pt'
                    dst       = os.path.join(weights_dir, new_name)
                    if fname != new_name:
                        if not os.path.exists(dst):
                            shutil.copy2(src, dst)   # create global-named file
                            renamed.append(f'{fname} → {new_name}')
                        else:
                            skipped.append(new_name) # already exists, don't overwrite
                        os.remove(src)               # always clean up local-named file
    if renamed:
        print(f"   Checkpoints renamed: {renamed}")
    if skipped:
        print(f"   Checkpoints skipped (already exist): {skipped}")

In [ ]:
# ──── Early-stop guard ────
epochs_no_improve, best_fitness = load_early_stop_state()
if epochs_no_improve >= PATIENCE:
    print(f"{'='*60}")
    print(f"   TRAINING STOPPED — Early stopping already triggered.")
    print(f"   ({epochs_no_improve} epochs with no improvement >= patience {PATIENCE})")
    print(f"   Best model: runs/detect/{run_name}/weights/best.pt")
    print(f"{'='*60}\n")
    raise SystemExit("Early stopping already triggered. Training is complete.")

# ──── Determine starting point ────
# Priority:
#   1 - chunk_state.yaml exists  → last.pt for weights, state file for epoch  (normal path)
#   2 - No state file, last.pt has valid epoch  → bootstrap from it  (migration from old setup)
#   3 - No state file, last.pt has epoch=-1  → fall back to highest periodic checkpoint
#   4 - Nothing found  → fresh start
last_epoch = load_chunk_state()

if last_epoch >= 0:
    current_epoch = last_epoch + 1
    if os.path.exists(checkpoint_path): # this is the most likely scenario since I set things up
        load_path = checkpoint_path
        print(f"   State file: last completed epoch = {last_epoch}")
        print(f"   Resuming from last.pt → starting epoch {current_epoch}")
    else: # if somehow last.pt has -1 epochs I am getting the highest epochs
        print(f"   State file found (epoch {last_epoch}) but last.pt is missing — searching")
        load_path = None
        if os.path.exists(weights_dir): # if weights_dir exists, search for the highest epoch checkpoint to resume from
            periodic = []
            for f in os.listdir(weights_dir):
                if f.startswith('epoch') and f.endswith('.pt'):
                    try:
                        periodic.append((int(f.replace('epoch','').replace('.pt','')), f))
                    except ValueError:
                        continue
            if periodic: # if I found any periodic checkpoints, get the one with the highest epoch number
                _, best_f = max(periodic, key=lambda x: x[0])
                load_path = os.path.join(weights_dir, best_f)
                print(f"   └─ Using: {best_f}")
            else:
                print(f"   └─ No checkpoints found. Starting fresh from base model.")

elif os.path.exists(checkpoint_path): # if last_epoch is not found but last.pt exists, try to bootstrap from it (this is for migration from old setup where I only had last.pt without the chunk_state.yaml)
    saved_epoch = get_checkpoint_epoch(checkpoint_path)
    if saved_epoch >= 0: # if last.pt has a valid epoch number, use it to resume training (migration from old setup)
        current_epoch = saved_epoch + 1
        load_path = checkpoint_path
        print(f"   No state file. Read epoch {saved_epoch} from last.pt.")
        print(f"   Resuming → starting epoch {current_epoch}")
    else:
        print(f"   last.pt has epoch=-1 (stripped) and no state file found.")
        print(f"   Searching for highest periodic checkpoint...")
        periodic = []
        for f in os.listdir(weights_dir):
            if f.startswith('epoch') and f.endswith('.pt'):
                try:
                    periodic.append((int(f.replace('epoch','').replace('.pt','')), f))
                except ValueError:
                    continue
        periodic.sort(reverse=True)
        print(f"   Found: {[f for _, f in periodic[:3]]}")
        if periodic: # if I found any periodic checkpoints, get the one with the highest epoch number
            best_n, best_f = periodic[0]
            load_path = os.path.join(weights_dir, best_f)
            saved_epoch = get_checkpoint_epoch(load_path)
            current_epoch = saved_epoch + 1
            print(f"   └─ Using: {best_f} (epoch {saved_epoch})")
        else:
            print(f"   └─ No periodic checkpoints found. Starting fresh.")
            current_epoch = 0
            load_path = None
else:
    current_epoch = 0
    load_path = None

next_target = min(current_epoch + EPOCHS_PER_CHUNK, TOTAL_EPOCHS)

if load_path: # if I have a checkpoint to load, load it, otherwise start fresh from the base model
    model = YOLO(load_path)
    print(f"   Loaded: {os.path.basename(load_path)}")
else:
    model = YOLO(base_model_path)
    print(f"   Starting fresh from: {base_model_path}")

print(f"   Training epochs {current_epoch} → {next_target} (+{next_target - current_epoch} epochs)")
print(f"{'='*60}\n")

   State file: last completed epoch = 225
   Resuming from last.pt → starting epoch 226
   Loaded: last.pt
   Training epochs 226 → 241 (+15 epochs)



In [ ]:
try:
    results = model.train(
        data='YAMLS/dataset.yaml', # dataset
        name=run_name, # set run name
        resume=False, # no resuming because it messes up the training, I am handling it myself
        exist_ok=True, 

        epochs=next_target - current_epoch, # my chunk
        patience=0, # custom patience tracking

        imgsz=640, # 640x640 my slice/background size
        batch=6,   # hardware limitations but gradient covers it, so the model acts like it is training on #x4 batch 

        cache='disk',  # ram cache is buggy with calculations, and my ssd is fast enough to handle it
        device='cuda',
        workers=4, # more than 6 causes bottlenecks on my hardware
        amp=True,

        save=True,
        save_period=5,

        val=True,
        plots=True,

        seed=69,
        deterministic=False,

        **training_config
    )

    gc.collect()
    torch.cuda.empty_cache()

    if results is not None and model.trainer is not None:
        # after training I need to update the epoch tracking and the early stop tracking based on the results of this chunk
        epochs_ran = model.trainer.epoch + 1 # this is the local epochs ran in this chunk
        actual_last_epoch = current_epoch + epochs_ran - 1 # this is the true epoch number based on the starting epoch and the local epochs ran
        save_chunk_state(actual_last_epoch) 
        fix_chunk_artifacts(current_epoch, next_target)
        current_fitness = model.trainer.best_fitness # this is the best fitness achieved in this chunk, I use it to update the early stop tracking

        if current_fitness > best_fitness: # if the fitness improved, reset the patience counter and update the best fitness
            save_early_stop_state(0, current_fitness)
            print(f"   Fitness improved: {best_fitness:.4f} → {current_fitness:.4f} — counter reset")
        else: # if no improvement, update the patience counter but keep the best fitness
            new_no_improve = epochs_no_improve + epochs_ran
            save_early_stop_state(new_no_improve, best_fitness)
            print(f"   No improvement ({new_no_improve}/{PATIENCE} patience used)")

        actual_total = actual_last_epoch + 1

    print(f"\n{'='*60}")
    if next_target >= TOTAL_EPOCHS:
        print(f"TRAINING FULLY COMPLETED! ({TOTAL_EPOCHS}/{TOTAL_EPOCHS} epochs)")
    else:
        print(f"   CHUNK COMPLETED! ({actual_total}/{TOTAL_EPOCHS} epochs)")
    print(f"   Best:  runs/detect/{run_name}/weights/best.pt")
    print(f"   Last:  runs/detect/{run_name}/weights/last.pt")
    print(f"{'='*60}\n")

except KeyboardInterrupt: # interrupted by me
    gc.collect()
    torch.cuda.empty_cache()
    print(f"\n{'='*60}")
    print(f"   INTERRUPTED — progress saved")
    print(f"{'='*60}\n")

# I probably fixed this by adding custom early stop tracking but I am not gonna risk it :)
except SystemExit: # this resolved a bug regarding early stopping I am actually not sure how 
    raise  # Let the early-stop guard propagate cleanly without hitting the error handler

except Exception as e: # any kind of other problem
    gc.collect()
    torch.cuda.empty_cache()
    print(f"\n{'='*60}")
    print(f"ERROR: {str(e)}")
    print(f"{'='*60}\n")
    raise

New https://pypi.org/project/ultralytics/8.4.22 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.19  Python-3.12.0 torch-2.9.1+cu128 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=6, bgr=0.0, box=12.0, cache=disk, cfg=None, classes=None, close_mosaic=10, cls=0.3, compile=False, conf=None, copy_paste=0.35, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=YAMLS/dataset.yaml, degrees=10.0, deterministic=False, device=0, dfl=2.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=15, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.5, hsv_v=0.3, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0008, lrf=0.1, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=runs/detect/yolo11m-medium-spec_AdamW/weights/last.pt, momentum=0.